In [1]:
import pandas as pd

# 1. Clean & align weather to Nepal Time
weather = pd.read_csv('Merged Weather.csv').drop(columns=['Unnamed: 0'])
weather['npt_time'] = pd.to_datetime(weather['valid_time']) + pd.Timedelta(hours=5, minutes=45)   # Shift to Nepal Time from UTC
weather['date'] = weather['npt_time'].dt.normalize()                                              # Strip the time while maintaining the
                                                                                                  # date to merge later
# 2. Aggregate 6-hourly -> Daily
daily_weather = weather.groupby('date').agg(            # Aggregate data from a given 24 hour period (4 x 6-hour readings)
    wind300_max=('max_wind_speed_300', 'max'),          # Worst case wind gust @300 (Near summit)
    wind300_mean=('max_wind_speed_300', 'mean'),        # Average wind gust @300
    wind400_max=('max_wind_speed_400', 'max'),          # Average wind gust @400 (Near EBC)
    temp300_min=('t_300', 'min'),                       # Worst case temp @300
    temp400_min=('t_400', 'min'),                       # Worst case temp @400
    z300_min=('z_300', 'min'),                          # Lowest geopotential reading of the day (Low indicating low pressure system / storm system)
    z300_range=('z_300', lambda x: x.max() - x.min()),  # Utilizing an inline lambd function to see how rapidly atmospheric pressure fluctated within the day
    rh300_mean=('r_300', 'mean'),                       # Average relative humidity @300
    tp_sum=('tp', 'sum'),                               # Total precipitation for the day
    tcc_max=('tcc', 'max')                              # Peak cloud cover for the day
).reset_index()

# 3. Rolling 3-day pre-summit features (shifted by 1 day so it's strictly prior days)
daily_weather = daily_weather.sort_values('date')
daily_weather['tp_prev_3d'] = daily_weather['tp_sum'].shift(1).rolling(3).sum()
daily_weather['wind300_prev_3d_max'] = daily_weather['wind300_max'].shift(1).rolling(3).max()
daily_weather['z300_24h_change'] = daily_weather['z300_min'].diff(1)

# 4. Merge onto Everest attempts
everest = pd.read_csv('everest_reduced.csv')
everest['smtdate'] = pd.to_datetime(everest['smtdate'])

df_merged = pd.merge(
    everest,
    daily_weather,
    left_on='smtdate',
    right_on='date',
    how='left'
).drop(columns=['date'])

In [2]:
df_merged.head()

,expid,year,smtdate,host,totmembers,tothired,success1,termreason,smtmembers,wind300_max,...,temp300_min,temp400_min,z300_min,z300_range,rh300_mean,tp_sum,tcc_max,tp_prev_3d,wind300_prev_3d_max,z300_24h_change
0,EVER95103,1995,1995-05-12,2,13,7,True,1,2,6.885866,...,246.01244,258.10120,95049.19,235.935,0.058332,0.000030,0.877533,0.000504,27.566714,204.63
1,EVER95104,1995,1995-05-14,2,20,14,True,1,8,5.135309,...,245.95332,261.45007,95064.31,202.880,0.216294,0.000168,0.970734,0.000173,9.480625,22.75
2,EVER95105,1995,1995-05-12,2,6,0,True,1,3,6.885866,...,246.01244,258.10120,95049.19,235.935,0.058332,0.000030,0.877533,0.000504,27.566714,204.63
3,EVER95106,1995,1995-05-11,2,23,2,True,1,7,5.666456,...,245.05974,256.86325,94844.56,259.500,0.217746,0.000038,0.870941,0.000704,32.850760,447.37
4,EVER95107,1995,1995-05-16,2,12,4,True,1,1,6.207080,...,247.13907,261.73870,94959.94,222.750,0.349006,0.000242,0.998657,0.000807,9.480625,21.38


In [3]:
try:
    df_merged.to_csv('Climbing-Weather Aligned.csv')
except: 
    raise('BufferError')